In [1]:
import os

In [2]:
%pwd

'c:\\Users\\Gauri\\Downloads\\Abhishek\\Doc-Summarization\\DocSummarizer\\research'

In [3]:
os.chdir("C:\\Users\\Gauri\\Downloads\\Abhishek\\Doc-Summarization\\DocSummarizer")

In [4]:
%pwd

'C:\\Users\\Gauri\\Downloads\\Abhishek\\Doc-Summarization\\DocSummarizer'

In [6]:
from pathlib import Path
from dataclasses import dataclass
@dataclass(frozen=True)
class DataTransformationConfig:
    raw_data_dir: Path
    data_path: Path
    transformed_data_dir: Path
    tokenizer_name: str


In [7]:


from DocSummarizer.constants import *
from DocSummarizer.utils.common import read_yaml

import os
import logging
from dataclasses import dataclass
from pathlib import Path

def create_directories_cust(dirs: list[str]):
    for dir_path in dirs:
        os.makedirs(dir_path, exist_ok=True)
        logging.info(f"Created directory: {dir_path}")


class ConfigurationManager:
    def __init__(self, config_file_path = CONFIG_FILE_PATH, params_file_path = PARAMS_FILE_PATH):
        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_file_path)

        # Ensure artifacts_root is a string
        artifacts_root = (
            self.config["artifacts_root"]
            if isinstance(self.config, dict)
            else self.config.artifacts_root
        )
        # create_directories(list([str(artifacts_root)]))
        print("artifacts_root type:", type(artifacts_root))
        print("artifacts_root value:", artifacts_root)
        create_directories_cust([artifacts_root])

    
    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories_cust([str(config.raw_data_dir)])
        
        data_transformation_config = DataTransformationConfig(
            raw_data_dir = Path(config.raw_data_dir),
            data_path = Path(config.data_path),
            transformed_data_dir = Path(config.transformed_data_dir),
            tokenizer_name = config.tokenizer_name
        )
        return data_transformation_config


In [ ]:
import os
import re
from DocSummarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset,load_from_disk

class DataTransformation:
    def __init__(self, config: DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name) 

    def convert_examples_to_features(example_batch):
        input_encodings=self.tokenizer(example_batch['dialogue'],max_length=1024,truncation=True)

  # Encode target summary (labels)
    target_encodings = self.tokenizer(
        text_target=example_batch["summary"],
        max_length=128,
        truncation=True,
        padding="max_length"
    )


     DataTransformation:
        def __init__(self, config: DataTransformationConfig):
            self.config = config
            self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

        def convert_examples_to_features(self, example_batch):
            input_encodings = self.tokenizer(
                example_batch['dialogue'],
                max_length=1024,
                truncation=True
            )

            target_encodings = self.tokenizer(
                text_target=example_batch["summary"],
                max_length=128,
                truncation=True,
                padding="max_length"
            )

            return {
                'input_ids': input_encodings['input_ids'],
                'attention_mask': input_encodings['attention_mask'],
                'labels': target_encodings['input_ids']
            }
